# Home Credit Default Risk - EDA
Full EDA pipeline.

In [ ]:
import os, sys
sys.path.insert(0, os.path.dirname(os.getcwd()))
os.environ.setdefault('DATA_DIR', '../data')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
DATA_DIR = os.getenv('DATA_DIR', '../data')
df = pd.read_csv(os.path.join(DATA_DIR, 'application_train.csv'))
print('Rows:', len(df), '  Cols:', df.shape[1])
print('Default rate:', round(df["TARGET"].mean()*100, 2), 'pct')
df.head()

## 2. Data Quality

In [ ]:
null_pct = df.isnull().mean().sort_values(ascending=False)
print('Columns with >30pct nulls:', (null_pct > 0.3).sum())
null_pct[null_pct > 0.3].head(10)

## 3. Target Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
tc = df['TARGET'].value_counts()
bars = ax.bar(['Repaid 0', 'Defaulted 1'], tc.values, color=['#3b82f6', '#ef4444'])
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 500, str(int(b.get_height())), ha='center')
ax.set_title('Loan Repayment Status')
plt.tight_layout()
plt.show()

## 4. Age Distribution

In [ ]:
df['age_years'] = (-df['DAYS_BIRTH']) / 365
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df[df['TARGET']==0]['age_years'], bins=40, alpha=0.6, color='#3b82f6', label='Repaid')
ax.hist(df[df['TARGET']==1]['age_years'], bins=40, alpha=0.6, color='#ef4444', label='Defaulted')
ax.set_title('Age Distribution by Default Status')
ax.set_xlabel('Age in years')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Default Rate by Education and Income Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ed = df.groupby('NAME_EDUCATION_TYPE')['TARGET'].mean().sort_values() * 100
axes[0].barh(ed.index, ed.values, color='#8b5cf6')
axes[0].set_xlabel('Default Rate pct')
axes[0].set_title('Default Rate by Education')
it = df.groupby('NAME_INCOME_TYPE')['TARGET'].mean().sort_values(ascending=False) * 100
axes[1].bar(it.index, it.values, color='#14b8a6')
axes[1].set_title('Default Rate by Income Type')
axes[1].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 6. Income Distribution

In [ ]:
cap = df['AMT_INCOME_TOTAL'].quantile(0.99)
df2 = df[df['AMT_INCOME_TOTAL'] < cap]
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df2[df2['TARGET']==0]['AMT_INCOME_TOTAL'], bins=50, alpha=0.6, color='#3b82f6', label='Repaid')
ax.hist(df2[df2['TARGET']==1]['AMT_INCOME_TOTAL'], bins=50, alpha=0.6, color='#ef4444', label='Defaulted')
ax.set_title('Income Distribution by Default Status')
ax.set_xlabel('Annual Income')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Correlation Heatmap

In [ ]:
cols = ['TARGET', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'DAYS_BIRTH', 'DAYS_EMPLOYED']
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df[cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 8. Key Business Insights

In [ ]:
print('Default rate pct:', round(df['TARGET'].mean()*100, 2))
print('Defaulter median income:', df[df['TARGET']==1]['AMT_INCOME_TOTAL'].median())
print('Non-defaulter median income:', df[df['TARGET']==0]['AMT_INCOME_TOTAL'].median())
print('Age 20-30 default pct:', round(df[df['age_years'].between(20,30)]['TARGET'].mean()*100, 2))
print('Higher-edu default pct:', round(df[df['NAME_EDUCATION_TYPE']=='Higher education']['TARGET'].mean()*100, 2))
print('Lower-sec default pct:', round(df[df['NAME_EDUCATION_TYPE']=='Lower secondary']['TARGET'].mean()*100, 2))